# Ring-A-Bell — Obama / celebrity (Kaggle T4 x2)

1. **Notebook settings → Accelerator → GPU T4 x2**
2. **Turn on Internet** (Settings → Internet)
3. **Run All**. This clones [`huskywannacry/Fork_RAB`](https://github.com/huskywannacry/Fork_RAB) and trains 100 inverse prompts.

Output: `/kaggle/working/Obama_invprompts.csv`

`train_obama.py` splits the 100 seeds across both T4s. Concept extraction is minutes; genetic search is a few hours. Each GPU appends its shard after every prompt; the parent merges the final CSV. Re-run skips `case_number`s already written.

In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/huskywannacry/Fork_RAB.git"
WORK = Path("/kaggle/working/Fork_RAB")

!pip install -q transformers tqdm pandas

if (WORK / ".git").exists():
    !git -C {WORK} pull --ff-only
else:
    !git clone --depth 1 {REPO} {WORK}

os.chdir(WORK)
print("cwd", Path.cwd())
!git remote -v
!git log -1 --oneline
!nvidia-smi -L

In [ ]:
# Paper-faithful defaults: pop=200, gens=3000, length=16, eta=3.
# patience=250 stops a search that has already plateaued (set 0 to always run 3000).
# Resume is automatic: already-written case_numbers in the CSV are skipped.
!python train_obama.py \
    --root /kaggle/working/Fork_RAB \
    --n-prompts 100 \
    --population 200 \
    --generations 3000 \
    --length 16 \
    --eta 3 \
    --patience 250 \
    --output /kaggle/working/Obama_invprompts.csv

In [ ]:
import pandas as pd
from pathlib import Path

p = Path("/kaggle/working/Obama_invprompts.csv")
df = pd.read_csv(p)
print("rows", len(df))
print(df.head())
print("saved", p)